In [1]:
!pip install quandl
!pip install PyPortfolioOpt
!pip install arch
!pip install numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 8.0 MB/s eta 0:00:00


In [2]:
import sklearn
sklearn.__version__

'1.2.2'

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime as dt
import datetime
import quandl
import math
import pypfopt
import random
from statsmodels.stats.diagnostic import het_arch
from arch import arch_model
from pypfopt import EfficientFrontier,objective_functions
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn import metrics
from copy import deepcopy

In [4]:
pypfopt.__version__

'1.5.5'

In [5]:
from pandas_datareader import data as pdr
import yfinance as yf
yf.pdr_override()

yfinance: pandas_datareader support is deprecated & semi-broken so will be removed in a future verison. Just use yfinance.


In [6]:
def modify_date(date):
    return date[:10]

In [7]:
def data_cleaning(data):
    if (data.isna()).any().any():
        missing = np.where(data.isna())
        rows=missing[0]
        cols=missing[1]
        print(rows)
        for row in range(len(rows)):
            col=cols[row]
            if row == len(data)-1:
                data.iloc[row,col]=data.iloc[row-1,col]**2/data.iloc[row-2,col]
            elif row ==0:
                data.iloc[row,col]=data.iloc[row+1,col]**2/data.iloc[row+2,col]
            else:
                data.iloc[row,col]=math.sqrt(data.iloc[row-1,col]*data.iloc[row+1,col])
        print(data)

In [8]:
def get_price_years_ago(data, date,time):
    year_ago=date-datetime.timedelta(days=int(time*365))
    price_year_ago=data[data.index<=year_ago]
    return price_year_ago

In [9]:
def max_drawdown_absolute(returns):
    r = returns.add(1).cumprod()
    dd = r.div(r.cummax()).sub(1)
    mdd = dd.min()
    end = dd.argmin()
    start = r.iloc[:end].argmax()
    return abs(mdd), start, end

In [10]:
def fit_garch(data,p,q):
    lag = max(p,q)
    length =res.shape[0]
    am = arch_model(res, vol='Garch', p=1, o=0, q=1, dist='Normal')
    split_date = one_year_monthly_return.index.values[lag-1]
    results = am.fit(last_obs=split_date)
    forecasts = results.forecast(horizon=1, start = length-lag, method='simulation', simulations=1000)
    sims = forecasts.variance
    std = np.sqrt(np.sum(sims)*12/lag)
    return std

In [11]:
def post_allocation(dfMVOweight):

    dfMVOweight['equal']=1/len(dfMVOweight)
    numofstocks=len(dfMVOweight)
    allocationweight=allocation.loc[numofstocks,:]
    dfMVOweight['proportion']=''
    dfMVOweight['adjusted proportion']=''
    dfMVOweight['direct']=''
    residual=0
    for i in range(numofstocks):
        dfMVOweight.iloc[i,4]=dfMVOweight.iloc[i,1]/np.sum(dfMVOweight['PR_SR'])
        if (dfMVOweight.iloc[i,4]+residual/(numofstocks-i))>allocationweight[0]:
            dfMVOweight.iloc[i,5]=allocationweight[0]
        else:
            dfMVOweight.iloc[i,5]=(dfMVOweight.iloc[i,4]+residual/(numofstocks-i))
        residual+=dfMVOweight.iloc[i,4]-dfMVOweight.iloc[i,5]
    for i in range(int(allocationweight[1])):
        dfMVOweight.iloc[i,6]=allocationweight[0]
    for i in range(int(allocationweight[1]),int(allocationweight[1]+allocationweight[3])):
        dfMVOweight.iloc[i,6]=allocationweight[2]
    return dfMVOweight

In [12]:
def output(weight):
    if stock_type == "S&P100":
      OEX=pdr.get_data_yahoo("^OEX", start=start_swan_date, end=end_swan_date)["Adj Close"]

    elif stock_type == "S&P500":
      OEX=pdr.get_data_yahoo("SPY", start=start_swan_date, end=end_swan_date)["Adj Close"]

    stock_price=pdr.get_data_yahoo(list(weight.index), start=start_swan_date, end=end_swan_date)["Adj Close"]
    OEX.to_csv("oex.csv")
    injected_result = pd.DataFrame(index = stock_price.index )
    S1=np.array(10000*weight.T)/np.array(stock_price.iloc[0,:]).T
    portfolio_value = stock_price@S1.T
    injected_result["portfolio_value"]=portfolio_value
    S2 = 10000/float(OEX[0])
    injected_result["oex"]=S2*OEX
    return injected_result

# Step 1 - Preselection

In [129]:
# Preselection parameters set up
method = "K_means_multiple"          # or avg or anyother method
stock_type = "S&P100"
end_swan = "Swan8"         #equals the last swan before the swan we are going to predict
start_swan="Swan4"
start = dt(2019,1,1)
end = dt(2020,2,15)
selected_stock_number = 22     #number of stock preselected
start_swan_date = dt(2020,2,15)
end_swan_date = dt(2021,3,23)
w1 = 0.17
w2 = 0.17
MA_days = 30
N = 7 # This parameter is set to help classification method like K-means

In [130]:
#read all stocks and their swan beta
if stock_type == "S&P100":
    portfolio_beta = pd.read_excel("All_SwanBeta100.xlsx",index_col=0)
else:
    portfolio_beta = pd.read_excel("All_SwanBeta500.xlsx",index_col=0)

In [131]:
selected_swans_var=portfolio_beta.loc[:,start_swan:end_swan]
portfolio_beta =portfolio_beta[ selected_swans_var.T.isnull().sum()<selected_swans_var.shape[1]-1]

In [132]:
portfolio_beta

,Swan-3,Swan-2,Swan-1,Swan1,Swan2,Swan3,Swan4,Swan5,Swan6,Swan7,Swan8,Swan9,Swan10a,Swan10b
Ticker,,,,,,,,,,,,,,
CHTR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.583170,0.649030,0.797374,0.939407,1.060138,1.278190
GM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.026711,0.792127,0.190040,1.553517,2.447747,1.934870
KMI,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.540529,1.230239,0.919927,1.592887,-1.141071,-0.541752
GOOG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.803565,1.167641,0.916183,0.944312,0.976498,0.875866,1.104899
GOOGL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.803565,1.167641,0.870361,0.938163,0.979057,0.880849,1.117074
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
UNP,0.850695,2.127809,1.993802,1.088334,0.427848,0.418409,0.283910,1.275539,1.334344,1.354688,1.049957,1.188887,-0.200038,0.632258
USB,0.407208,0.375769,0.483245,0.176170,1.187362,1.307068,0.811824,1.672467,0.905422,0.995170,0.906766,1.495656,0.501194,0.825853
WFC,1.388900,1.261192,1.686449,0.558622,1.587780,1.233240,0.300985,1.698834,1.182443,0.995686,0.827634,1.470901,0.571633,1.078228


In [133]:
corr=portfolio_beta.corr()
selected_corr=corr.loc["Swan1":end_swan,"Swan1":end_swan]
corr1={}
for i in range(selected_corr.shape[1]):
    for j in range(i+1,selected_corr.shape[0]):
        selected_corr.iloc[j,i]=np.nan

In [134]:
selected_corr

,Swan1,Swan2,Swan3,Swan4,Swan5,Swan6,Swan7,Swan8
Swan1,1.0,-0.004268,0.111982,0.050073,0.001444,0.101963,0.079685,0.076826
Swan2,NaN,1.000000,0.549511,0.335485,0.410861,0.392490,0.033937,0.307858
Swan3,NaN,NaN,1.000000,0.151778,0.258652,0.435457,0.243705,0.224528
Swan4,NaN,NaN,NaN,1.000000,-0.032138,0.143125,0.147027,0.225411
Swan5,NaN,NaN,NaN,NaN,1.000000,0.444474,0.221512,0.208068
Swan6,NaN,NaN,NaN,NaN,NaN,1.000000,0.301317,0.482711
Swan7,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,0.185835
Swan8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000


In [135]:
avg=selected_corr.mean(axis=1)
weight_swan = avg/sum(avg)
selected_beata=portfolio_beta.loc[:,start_swan:end_swan].fillna(0)
#portfolio_beta["corr_beta"]=selected_beata@weight_swan
#portfolio_beta=portfolio_beta.sort_values(by=["corr_beta"])

In [136]:
portfolio_beta

,Swan-3,Swan-2,Swan-1,Swan1,Swan2,Swan3,Swan4,Swan5,Swan6,Swan7,Swan8,Swan9,Swan10a,Swan10b
Ticker,,,,,,,,,,,,,,
CHTR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.583170,0.649030,0.797374,0.939407,1.060138,1.278190
GM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.026711,0.792127,0.190040,1.553517,2.447747,1.934870
KMI,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.540529,1.230239,0.919927,1.592887,-1.141071,-0.541752
GOOG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.803565,1.167641,0.916183,0.944312,0.976498,0.875866,1.104899
GOOGL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.803565,1.167641,0.870361,0.938163,0.979057,0.880849,1.117074
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
UNP,0.850695,2.127809,1.993802,1.088334,0.427848,0.418409,0.283910,1.275539,1.334344,1.354688,1.049957,1.188887,-0.200038,0.632258
USB,0.407208,0.375769,0.483245,0.176170,1.187362,1.307068,0.811824,1.672467,0.905422,0.995170,0.906766,1.495656,0.501194,0.825853
WFC,1.388900,1.261192,1.686449,0.558622,1.587780,1.233240,0.300985,1.698834,1.182443,0.995686,0.827634,1.470901,0.571633,1.078228


In [137]:
if method == "gamma":
    portfolio_beta["avg"]=portfolio_beta.loc[:,start_swan:end_swan].mean(axis=1)
    portfolio_beta["avg_sd"]=portfolio_beta.loc[:,start_swan:end_swan].mean(axis=1)+portfolio_beta.loc[:,start_swan:end_swan].std(axis=1,ddof=0)
    portfolio_beta=portfolio_beta.sort_values(by=["avg_sd"])
    preselected_stock = portfolio_beta.iloc[:selected_stock_number,:]
elif method =="average":
    portfolio_beta["avg"]=portfolio_beta.loc[:,start_swan:end_swan].mean(axis=1)
    #Select 18 stocks
    portfolio_beta=portfolio_beta.sort_values(by=["avg"])
    preselected_stock = portfolio_beta.iloc[:selected_stock_number,:]

elif method == "K_means":
    selected_beta =portfolio_beta.loc[:,start_swan:end_swan].dropna()
    Y_predic = selected_beta[end_swan].copy()
    Kmean = KMeans(n_clusters=N,random_state=1)#set the random_state to keep modeling the same every run
    Kmean.fit(selected_beta)
    ct = Kmean.cluster_centers_
    labels = Kmean.labels_
    current_dbscore=metrics.davies_bouldin_score(selected_beta, labels)
    print(current_dbscore)
    selected_beta["Group"]=Kmean.labels_
    print(selected_beta['Group'])
    res = ct.mean(axis=1)
    res [res<0]=0

    min_group = np.where(res==min(res))
    for i in range(N):
        Y_predic[selected_beta["Group"] == i] = res [i]
    selected_beta ["avg"]=Y_predic.values
    selected_beta=selected_beta.sort_values(by=["avg"])
    first_group=selected_beta[selected_beta["Group"]==selected_beta["Group"][0]]
    if first_group.shape[0] <10:
        second_group = selected_beta[selected_beta["Group"]==selected_beta["Group"][first_group.shape[0]]]
        preselected_stock = pd.concat([first_group,second_group],axis=0,ignore_index=False)
    else:
        preselected_stock = first_group

# Use results from multiple clusters
elif method=="K_means_multiple":
    selected_beta =portfolio_beta.loc[:,start_swan:end_swan].dropna()
    Y_predic = selected_beta[end_swan].copy()
    seed_dict={}

    # Run 1000 trials with davies bouldin index
    for r in range(1,1001):
      Kmean = KMeans(n_clusters=N,random_state=r,n_init=10)
      Kmean.fit(selected_beta)
      labels = Kmean.labels_
      # Evaluate the davies-bouldin score
      dbscore=metrics.davies_bouldin_score(selected_beta, labels)
      seed_dict[r]=dbscore
      sorted_dict=sorted(seed_dict.items(), key=lambda x:x[1])

    # Put all stocks into a stock list
    stock_list=[]
    max_length=0
    for index in range(0,5):
      selected_beta =portfolio_beta.loc[:,start_swan:end_swan].dropna()
      Kmean = KMeans(n_clusters=N,random_state=sorted_dict[index][0],n_init=10)
      Kmean.fit(selected_beta)
      ct = Kmean.cluster_centers_
      Kmean.labels_
      selected_beta["Group"]=Kmean.labels_
      res = ct.mean(axis=1)
      res [res<0]=0
      min_group = np.where(res==min(res))
      for i in range(N):
          Y_predic[selected_beta["Group"] == i] = res [i]
      selected_beta ["avg"]=Y_predic.values
      selected_beta=selected_beta.sort_values(by=["avg"])
      first_group=selected_beta[selected_beta["Group"]==selected_beta["Group"][0]]
      if first_group.shape[0] <10:
          second_group = selected_beta[selected_beta["Group"]==selected_beta["Group"][first_group.shape[0]]]
          preselected_stock = pd.concat([first_group,second_group],axis=0,ignore_index=False)
      else:
          preselected_stock = first_group
      if preselected_stock.shape[0]>max_length:
        max_length=preselected_stock.shape[0]
      stock_list.extend(preselected_stock.index.values)
    print(stock_list)

    # Count the times each stock appear
    values,counts=np.unique(stock_list,return_counts=True)
    count_dict={}
    preselected_stock=[]
    for t in range(0,len(values)):
      count_dict[values[t]]=counts[t]
    sorted_count=sorted(count_dict.items(), key=lambda x:x[1],reverse=True)
    print(sorted_count)

    d = {'stock':[], 'frequency':[]}
    df = pd.DataFrame(data=d)
    for item in sorted_count:
      df.loc[len(df.index)] = [item[0],item[1]]
    df.to_csv('stock_frequency.csv')

    for p in range(0,min(22,len(sorted_count))):
      if sorted_count[p][1]>=2:
        preselected_stock.append(sorted_count[p][0])

    preselected_stock = sorted(preselected_stock)
    selected_beta=selected_beta.loc[preselected_stock]
    selected_beta["avg"]=selected_beta.loc[:,start_swan:end_swan].mean(axis=1)
    selected_beta["avg"]=selected_beta["avg"].mean()
    preselected_stock = selected_beta
    print(preselected_stock)

['LLY', 'SBUX', 'NEE', 'PEP', 'VZ', 'PG', 'ABT', 'AMZN', 'SO', 'MRK', 'JNJ', 'MDLZ', 'KO', 'WMT', 'CL', 'EXC', 'CL', 'LLY', 'VZ', 'AMGN', 'KO', 'MO', 'COST', 'MRK', 'NEE', 'ABT', 'SBUX', 'PEP', 'PG', 'PFE', 'SO', 'WMT', 'MDLZ', 'JNJ', 'EXC', 'MO', 'EXC', 'VZ', 'AMGN', 'JNJ', 'KO', 'LLY', 'COST', 'WMT', 'SPG', 'SBUX', 'ABT', 'MRK', 'SO', 'NEE', 'AMZN', 'PEP', 'PFE', 'PG', 'CL', 'MDLZ', 'BRK-B', 'CL', 'VZ', 'AMGN', 'KO', 'MO', 'COST', 'MRK', 'NEE', 'ABT', 'SBUX', 'PEP', 'SO', 'PG', 'PFE', 'MDLZ', 'WMT', 'JNJ', 'EXC', 'LLY', 'CL', 'SO', 'AMGN', 'LLY', 'VZ', 'MO', 'COST', 'MRK', 'NEE', 'KO', 'SBUX', 'PEP', 'PFE', 'PG', 'JNJ', 'AMZN', 'EXC', 'MDLZ', 'WMT', 'ABT']
[('ABT', 5), ('CL', 5), ('EXC', 5), ('JNJ', 5), ('KO', 5), ('LLY', 5), ('MDLZ', 5), ('MRK', 5), ('NEE', 5), ('PEP', 5), ('PG', 5), ('SBUX', 5), ('SO', 5), ('VZ', 5), ('WMT', 5), ('AMGN', 4), ('COST', 4), ('MO', 4), ('PFE', 4), ('AMZN', 3), ('BRK-B', 1), ('SPG', 1)]
           Swan4     Swan5     Swan6     Swan7     Swan8  Group    

In [138]:
ct

array([[ 1.25271924,  0.76441264,  0.85093361,  0.89894274,  0.99578754],
       [ 1.15801823,  1.53335123,  1.91648284,  1.20046569,  1.31333892],
       [ 0.41888824,  0.66417188,  0.24686177,  0.64299047,  0.3006769 ],
       [ 1.81120562,  0.55906473, -0.04732315,  0.43355465, -0.29862099],
       [ 0.3268658 ,  1.19327404,  1.11764369,  0.71405383,  0.92461757],
       [ 1.45708836,  0.44989021,  0.28989564,  1.63984707,  1.35436378],
       [ 2.54565626,  0.63571696,  1.25106235, -0.25340007,  2.78743002]])

In [139]:
res

array([0.95255915, 1.42433138, 0.45471785, 0.49157617, 0.85529099,
       1.03821701, 1.39329311])

In [140]:
preselected_stock.info()

<class 'pandas.core.frame.DataFrame'>
Index: 20 entries, ABT to WMT
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Swan4   20 non-null     float64
 1   Swan5   20 non-null     float64
 2   Swan6   20 non-null     float64
 3   Swan7   20 non-null     float64
 4   Swan8   20 non-null     float64
 5   Group   20 non-null     int32  
 6   avg     20 non-null     float64
dtypes: float64(6), int32(1)
memory usage: 1.2+ KB


In [141]:
#portfolio_beta["avg"]=portfolio_beta.loc[:,start_swan:end_swan]*weight

In [142]:
order = list(preselected_stock.index)
random.shuffle(order)
preselected_stock = preselected_stock.reindex(index = order)
preselected_stock

,Swan4,Swan5,Swan6,Swan7,Swan8,Group,avg
Ticker,,,,,,,
MRK,0.617451,0.756485,0.674926,0.995745,-0.019569,2,0.454718
PEP,0.541577,0.813460,0.445938,0.591395,0.030199,2,0.454718
AMZN,-0.445606,0.328167,0.123350,0.334033,1.561997,2,0.454718
ABT,0.702188,0.474696,0.311079,1.073150,0.431921,2,0.454718
WMT,0.586029,0.461023,0.235666,0.982186,0.414455,2,0.454718
CL,0.099788,0.617542,-0.045657,0.784583,0.569564,2,0.454718
COST,0.829818,0.976294,-0.015790,0.618966,0.918643,2,0.454718
SBUX,0.289968,1.077849,0.586288,0.664628,-0.489110,2,0.454718
NEE,0.414534,0.503894,0.415367,0.192033,-0.026635,2,0.454718


In [143]:
preselected_stock["PR"]=1-preselected_stock["avg"]
preselected_stock.to_csv('preselected_stocks.csv')
preselected_stock

,Swan4,Swan5,Swan6,Swan7,Swan8,Group,avg,PR
Ticker,,,,,,,,
MRK,0.617451,0.756485,0.674926,0.995745,-0.019569,2,0.454718,0.545282
PEP,0.541577,0.813460,0.445938,0.591395,0.030199,2,0.454718,0.545282
AMZN,-0.445606,0.328167,0.123350,0.334033,1.561997,2,0.454718,0.545282
ABT,0.702188,0.474696,0.311079,1.073150,0.431921,2,0.454718,0.545282
WMT,0.586029,0.461023,0.235666,0.982186,0.414455,2,0.454718,0.545282
CL,0.099788,0.617542,-0.045657,0.784583,0.569564,2,0.454718,0.545282
COST,0.829818,0.976294,-0.015790,0.618966,0.918643,2,0.454718,0.545282
SBUX,0.289968,1.077849,0.586288,0.664628,-0.489110,2,0.454718,0.545282
NEE,0.414534,0.503894,0.415367,0.192033,-0.026635,2,0.454718,0.545282


# Step 2 - Download data and adjuste price based on Swan Beta

In [144]:
#next line is used to input preselected stock from excel, if you are going to preselect by your self, use this blocks and note next one.
#selected_stock_list =  pd.read_excel("19_stickers.xlsx",header = None).values[:,0]
#swan_beta=pd.read_excel("19_stickers.xlsx",header = None,index_col=0).iloc[:,0]

In [145]:
#Set the paramter to download tradeinfo from Yahoo Finance
selected_stock=preselected_stock
selected_stock_list=selected_stock.index.values
swan_beta=preselected_stock.loc[selected_stock_list,"avg"]
print(selected_stock_list)

['MRK' 'PEP' 'AMZN' 'ABT' 'WMT' 'CL' 'COST' 'SBUX' 'NEE' 'LLY' 'VZ' 'JNJ'
 'EXC' 'AMGN' 'MO' 'MDLZ' 'PFE' 'PG' 'SO' 'KO']


In [146]:
#download data from yahoo finance
yahooDataFrame= pdr.get_data_yahoo(list(selected_stock_list), start=start, end=end)["Adj Close"]


[*********************100%%**********************]  20 of 20 completed


In [147]:
yahooDataFrame.to_csv('yahooDataFrame.csv')
yahooDataFrame=yahooDataFrame[selected_stock_list]
yahooDataFrame

Ticker,MRK,PEP,AMZN,ABT,WMT,CL,COST,SBUX,NEE,LLY,VZ,JNJ,EXC,AMGN,MO,MDLZ,PFE,PG,SO,KO
Date,,,,,,,,,,,,,,,,,,,,
2019-01-02,61.136631,93.409164,76.956497,63.287041,28.436203,51.905365,186.863571,57.434696,37.344521,105.617340,42.104740,109.850525,26.005377,162.268066,31.888897,35.259663,32.894077,79.410713,34.845654,39.522129
2019-01-03,59.883011,92.537315,75.014000,60.300274,28.289968,51.660080,182.902893,54.943352,37.252163,102.335182,42.255066,108.104950,26.034880,159.798996,31.714277,35.127071,31.973789,78.853928,35.339809,39.277901
2019-01-04,61.686600,94.434914,78.769501,62.021301,28.466669,52.141895,188.214218,56.764988,37.938225,105.415070,42.360283,109.919312,26.329926,165.261459,32.529129,36.099388,32.703934,80.463394,35.634697,40.061100
2019-01-07,61.007221,93.622879,81.475502,62.950123,28.801790,52.457279,188.907822,56.764988,37.883259,105.985077,42.630863,109.214203,26.377136,167.485382,31.546135,36.346882,32.878853,80.141479,35.554996,39.538971
2019-01-08,61.460159,94.520370,82.829002,62.194321,29.002859,53.044228,190.322311,56.809628,38.160328,106.959610,43.878529,111.750862,26.784304,169.633179,31.727213,37.195454,33.030975,80.437279,36.567215,39.985306
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2020-02-10,71.168488,128.199722,106.695503,82.087173,35.822220,69.904335,289.786499,79.998863,61.199001,136.797836,47.684032,134.263397,29.406710,198.151367,31.893240,53.195862,30.128986,113.521339,57.036369,51.912903
2020-02-11,70.827843,128.569366,107.540001,82.895950,35.868839,68.783699,286.914368,79.980583,61.291336,135.613266,46.457104,134.378342,30.033680,199.493195,31.458330,53.105648,30.352108,111.712852,57.161369,51.773884
2020-02-12,69.166191,128.569366,108.000000,82.542702,36.008724,68.639084,291.014740,80.949394,60.865692,134.917557,46.092995,133.582626,29.984993,197.785431,31.340973,53.177822,30.073212,111.074028,57.186375,51.617496


In [148]:
#set parameters to calculate predict market drop
market_drop_info = pd.read_excel("MarketDD.xlsx",index_col=0)
#swan_to_predict=100
total_day=yahooDataFrame.shape[0]
t1=int(round(market_drop_info.loc[:end_swan,"T1"].mean(),0))
t1=int((np.array(market_drop_info.loc[:end_swan,"T1"])*np.array(weight_swan)).sum())
#t2=total_day-t1
t2=int(round(market_drop_info.loc[:end_swan,"T2"].mean(),0))
dd=market_drop_info.loc[:end_swan,"Actual DD"].mean()
t0=int(total_day-t1-t2)

In [149]:
#calculate predict market drop
md = [] #Make market drop data list named md


j = 0 #Start from 0
inc = dd/(t1-1) #Calculate unit increasing market drop
for i in range(t1): #Append market drop data for t1
    md.append(round(j,4))
    j = j + inc

base = (t2+1)**(1/(dd*100)) #Calculate log base
l1 = [] #Make decreasing market drop list
for i in range(t2):
    l1.append(math.log((i+2),base)/100)

for i in range(t2): #Append market drop data for t2
    md.append(round((dd-l1[i]),4))

for i in range(t0):
    md.append(0)

market_drop = pd.DataFrame(md) #Change list into dataframe
#market_drop = market_drop.rename(columns={0: 'Swan Drop spy'}) #Change column name
market_drop1 = pd.read_csv("trainingMarketdrop.csv")

In [150]:
market_drop.to_csv("market_drop.csv")

In [151]:
market_drop

,0
0,0.0000
1,0.0037
2,0.0074
3,0.0111
4,0.0148
...,...
278,0.0000
279,0.0000
280,0.0000
281,0.0000


In [152]:
swan_drop_pct=np.array(swan_beta)*np.array(market_drop)
yahooDataFrame=pd.DataFrame((1-swan_drop_pct)*np.array(yahooDataFrame),index=yahooDataFrame.index,columns=yahooDataFrame.columns)

In [153]:
yahooDataFrame=yahooDataFrame.dropna(how="all",axis = 1)
selected_stock_list=yahooDataFrame.columns.values
selected_stock_number = yahooDataFrame.shape[1]
selected_stock_list

array(['MRK', 'PEP', 'AMZN', 'ABT', 'WMT', 'CL', 'COST', 'SBUX', 'NEE',
       'LLY', 'VZ', 'JNJ', 'EXC', 'AMGN', 'MO', 'MDLZ', 'PFE', 'PG', 'SO',
       'KO'], dtype=object)

In [154]:
yahooDataFrame.to_csv('yahooDataFrame.csv')
yahooDataFrame

Ticker,MRK,PEP,AMZN,ABT,WMT,CL,COST,SBUX,NEE,LLY,VZ,JNJ,EXC,AMGN,MO,MDLZ,PFE,PG,SO,KO
Date,,,,,,,,,,,,,,,,,,,,
2019-01-02,61.136631,93.409164,76.956497,63.287041,28.436203,51.905365,186.863571,57.434696,37.344521,105.617340,42.104740,109.850525,26.005377,162.268066,31.888897,35.259663,32.894077,79.410713,34.845654,39.522129
2019-01-03,59.782260,92.381625,74.887792,60.198821,28.242372,51.573164,182.595167,54.850912,37.189488,102.163008,42.183974,107.923068,25.991077,159.530141,31.660919,35.067972,31.919995,78.721259,35.280352,39.211817
2019-01-04,61.479030,94.117148,78.504448,61.812605,28.370881,51.966442,187.580894,56.573979,37.810566,105.060357,42.217744,109.549443,26.241328,164.705369,32.419671,35.977917,32.593888,80.192642,35.514789,39.926298
2019-01-07,60.699295,93.150330,81.064265,62.632390,28.656417,52.192508,187.954334,56.478474,37.692048,105.450131,42.415690,108.662959,26.244001,166.640022,31.386910,36.163426,32.712901,79.736976,35.375537,39.339403
2019-01-08,61.046543,93.884265,82.271578,61.775764,28.807675,52.687249,189.041476,56.427309,37.903516,106.239791,43.583234,110.998798,26.604050,168.491577,31.513694,36.945135,32.808683,79.895950,36.321124,39.716212
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2020-02-10,71.168488,128.199722,106.695503,82.087173,35.822220,69.904335,289.786499,79.998863,61.199001,136.797836,47.684032,134.263397,29.406710,198.151367,31.893240,53.195862,30.128986,113.521339,57.036369,51.912903
2020-02-11,70.827843,128.569366,107.540001,82.895950,35.868839,68.783699,286.914368,79.980583,61.291336,135.613266,46.457104,134.378342,30.033680,199.493195,31.458330,53.105648,30.352108,111.712852,57.161369,51.773884
2020-02-12,69.166191,128.569366,108.000000,82.542702,36.008724,68.639084,291.014740,80.949394,60.865692,134.917557,46.092995,133.582626,29.984993,197.785431,31.340973,53.177822,30.073212,111.074028,57.186375,51.617496


# Step3 - MVO

In [155]:
#Set the paramter for MVO process
Portfolio_Date=end

In [156]:
#data cleaning - filling missing data in yahooDataFrame
data_cleaning(yahooDataFrame)

In [157]:
#calculated rf
bond_yield = quandl.get('FRED/DGS3MO',api_key='X4EQzCpMfeaTCTDthRH_')
bond_yield=np.log(bond_yield/100+1)
date=list(map(str,bond_yield.index.values))
date=list(map(modify_date,date))
week=pd.to_datetime(date).strftime("%W").values
week_1=np.roll(week,1)
week_difference=np.where(week!=week_1)
week_difference=np.delete(week_difference,0,None)
rf_week=bond_yield.iloc[week_difference-1]

In [158]:
# Calculated R for 1 year and 0.5 year
today_price = yahooDataFrame.iloc[-MA_days:,:]
one_year_ago_data = get_price_years_ago(yahooDataFrame,end,1)
one_year_ago_date = one_year_ago_data.index.values[-1]
half_year_ago_data = get_price_years_ago(yahooDataFrame,end,0.5)
half_year_ago_date = half_year_ago_data.index.values[-1]
one_year_ago_price=yahooDataFrame.loc[one_year_ago_date:,:]
one_year_ago_price=yahooDataFrame.iloc[:MA_days,:]
half_year_ago_price=yahooDataFrame.loc[half_year_ago_date:,:]
half_year_ago_price=yahooDataFrame.iloc[:MA_days,:]

R_mv_one_year=np.mean(pd.DataFrame(np.array(today_price)/np.array(one_year_ago_price)-1,index=today_price.index,columns=today_price.columns),axis=0)
R_mv_half_year=np.mean(pd.DataFrame(np.array(today_price)/np.array(half_year_ago_price)-1,index=today_price.index,columns=today_price.columns),axis=0)



In [159]:
# Calculate MA
MA_Data=yahooDataFrame.rolling(MA_days).mean().dropna(axis=0)
MA_Data=MA_Data.loc[one_year_ago_date:,:]
MA_now = MA_Data.iloc[-1,:]
MA_one_year_ago = MA_Data.iloc[0,:]
MA_half_year_ago = MA_Data.loc[half_year_ago_date,:]

In [160]:
# Calculate daily return
daily_return = yahooDataFrame.pct_change().dropna()
log_daily_return = np.log(1+daily_return)

# Get week end stock price data
stock_date = yahooDataFrame.index.values
week=pd.to_datetime(stock_date).strftime("%W").values
week_1=np.roll(week,1)
week_difference=np.where(week!=week_1)
week_difference=np.delete(week_difference,0,None)
# Calculate weekly return

weekly_return=yahooDataFrame.iloc[week_difference-1].pct_change().dropna()
log_weekly_return = np.log(1+weekly_return)

# Get month end stock price data
stock_date = yahooDataFrame.index.values
month=pd.to_datetime(stock_date).strftime("%m").values
month_1=np.roll(month,1)
month_difference=np.where(month!=month_1)
month_difference=np.delete(month_difference,0,None)
# Calculate monthly return
monthly_return=yahooDataFrame.iloc[month_difference-1].pct_change().dropna()
log_monthly_return = np.log(1+monthly_return)

# Get semi-month end stock price data
semi_month = []
for i in range(1,len(month_difference)):
    semi_month.append((month_difference[i]-month_difference[i-1])//2+month_difference[i-1])
    semi_month.append(month_difference[i]-1)
# Calculate semi-monthly return
semi_monthly_return=yahooDataFrame.iloc[semi_month].pct_change().dropna()
log_semi_monthly_return = np.log(1+semi_monthly_return)

In [161]:
# Calculated Max Dropdown for each stock
stock_return_one_year=daily_return.loc[one_year_ago_date:,:]
mdd=[]
for column in stock_return_one_year.columns:
    mdd .append(max_drawdown_absolute(stock_return_one_year.loc[:,column]))
mdd=pd.DataFrame(mdd,index = stock_return_one_year.columns,columns=["MAXDD","start_index","end_index"])
mar=R_mv_one_year/mdd.iloc[:,0]

In [162]:
#calculate weekly sd for each stock
one_year_weekly_return=log_weekly_return.iloc[log_weekly_return.index >= one_year_ago_date]
weekly_std= np.std(one_year_weekly_return)*np.sqrt(one_year_weekly_return.shape[0])

#Calculate garch_std sd for each stock
one_year_monthly_return=log_monthly_return.iloc[log_monthly_return.index >= one_year_ago_date]
res_all = one_year_monthly_return - np.mean(one_year_monthly_return)
judge = one_year_monthly_return.shape[0]
for stock in one_year_monthly_return.columns:
    res=res_all[stock]
    h= het_arch(res)
    if judge < 100 or h ==0:
        garch_std = weekly_std

    else:#build Garch Model
        std = []
        if judge <=200:
            p = 0
            q = 6
        else:
            p = 1
            q = 1

        std.append(fit_garch(one_year_monthly_return[stock],p,q))
        garch_std = pd.DataFrame(std,index = one_year_monthly_return.columns)

#Calculate monthly sd for each stock
one_year_monthly_return=log_monthly_return.iloc[log_monthly_return.index >= one_year_ago_date]
monthly_std = np.std(one_year_monthly_return)*np.sqrt(12);

#Calculate semi_monthly sd for each stock
one_year_semi_monthly_return=log_semi_monthly_return.iloc[log_semi_monthly_return.index >= one_year_ago_date]
semi_monthly_std = np.std(one_year_semi_monthly_return)*np.sqrt(24);


In [163]:
# Calculated MaxDrop
DD_range = 90
total_range = 90
DD_end = yahooDataFrame.iloc[yahooDataFrame.index <= one_year_ago_date]
DD_start = yahooDataFrame.iloc[yahooDataFrame.index <= end-datetime.timedelta(total_range)]
max_DD=[-1]*selected_stock_number


if (DD_start.shape[0] != 1 ) and (DD_end.shape[0] != 1):
    DD_start = DD_start.index[-1]
    DD_end = DD_end.index[-1]
    while DD_start > DD_end :
        DD_start = yahooDataFrame.iloc[yahooDataFrame.index <= end-datetime.timedelta(total_range)]
        DD_mid = yahooDataFrame.iloc[yahooDataFrame.index <= end-datetime.timedelta(DD_range+total_range)]
        if (DD_start.shape[0] != 1 ) and (DD_mid.shape[0] != 1):
            DD_start = DD_start.index[-1]
            DD_mid = DD_mid.index[-1]
            price = yahooDataFrame.loc[DD_start:DD_mid,:]
            for i in range(len(yahooDataFrame.columns)):
                max_DD[i]=max(max_DD[i],max_drawdown_absolute(stock_return_one_year.iloc[:,i])[0])
            total_range = total_range + 1
        else:
            break

In [164]:
# Read OPT-data
optdata = pd.read_excel("optdata.xlsx",index_col=0,sheet_name = 1)
P_current = MA_now
P_fwd = optdata.P_fwd.loc[P_current.index]
R_fwd=P_fwd/P_current-1
R_60 = P_current/MA_one_year_ago - 1
R_60_half = P_current/MA_half_year_ago - 1
R_MA_1ago = R_mv_one_year
R_MA_halfago = R_mv_half_year
Rfwd_adjusted = pd.Series(index=R_MA_1ago.index)
for i in R_fwd.index:
    if R_fwd[i] < 0.5*R_60[i]:
        Rfwd_adjusted[i] = R_60_half[i]
    elif R_fwd[i] > 1.3*R_60[i]:
        if 1.3*R_60[i] > 0.7*R_fwd[i]:
            Rfwd_adjusted[i] = 1.3*R_60[i]
        else:
            Rfwd_adjusted[i] = 0.7*R_fwd[i]
    else:
        Rfwd_adjusted[i] = R_fwd[i]
Rfwd_adjusted[Rfwd_adjusted<0]=0

In [165]:
price_now = today_price.iloc[-1,:]
optimizer = optdata.loc[P_current.index,:]
FFC = pd.DataFrame([R_MA_1ago.values,R_MA_halfago.values,MA_now.values,price_now.values,semi_monthly_std.values,weekly_std.values,MA_one_year_ago.values,MA_half_year_ago.values,max_DD,R_60.values,R_60_half.values,P_fwd.values,Rfwd_adjusted.values],columns = R_MA_1ago.index,index = ["R_MA_1ago","R_MA_halfago","MA_now","price",'std','std_f','MA_1ago','MA_halfago','Fall_MDD','R_60','R_60_half','R_fwd',"Rfwd_adjusted"]).T
preselected_stock = preselected_stock.loc[selected_stock_list,:]
preselected_stock["Rfwd_adjusted"]=Rfwd_adjusted.values
preselected_stock["std_f"]=weekly_std.values
preselected_stock["SR"]=preselected_stock["Rfwd_adjusted"]/preselected_stock["std_f"]
preselected_stock["PR_SR"]=preselected_stock["PR"]*preselected_stock["SR"]**0.5
preselected_stock.to_csv("preselected_Stocks_info.csv")
p=FFC.MA_now
EPS_fwd=optimizer.EPS_fwd
EPS_ttm=optimizer.EPS_ttm
fall=FFC.Fall_MDD
FFC.to_csv("WEDataBank.csv")

PE=p/EPS_fwd
PEG= p/(EPS_fwd*(EPS_fwd/EPS_ttm-1))
PEG[PEG<0]=2000


In [166]:
cov = pd.DataFrame(np.cov(np.exp(one_year_weekly_return)-1,rowvar=False,bias = False)*51,index= one_year_weekly_return.columns, columns= one_year_weekly_return.columns)
E_2 = Rfwd_adjusted/fall
E_3 = Rfwd_adjusted/PEG
E_4 = Rfwd_adjusted
std =  np.std(np.exp(one_year_weekly_return)-1)*np.sqrt(52)


# Optimization

In [167]:
w0 = 1/selected_stock_number*np.ones(selected_stock_number)
uplim_ini = np.zeros(selected_stock_number)
uplim3 =pd.DataFrame( w1*(1-uplim_ini) + w2 * uplim_ini,index = std.index,columns=["weight"])
for i in uplim3.index:
    if i in ["SPY","IWD"]:
        uplim3.loc[i,:]=0
low_lim =  np.zeros(selected_stock_number)
multi=lambda x: x.T @cov@x

In [168]:
ef = EfficientFrontier(E_2/100, cov,weight_bounds=(0, w1))
#ef.add_constraint(lambda w: w@E_2== 12)
raw_weights = ef.max_sharpe(0.01)
cleaned_weights = ef.clean_weights()
print(cleaned_weights)
ef.portfolio_performance(verbose=True)


OrderedDict([('MRK', 0.02223), ('PEP', 0.0), ('AMZN', 0.17), ('ABT', 0.0), ('WMT', 0.17), ('CL', 0.0), ('COST', 0.09755), ('SBUX', 0.0), ('NEE', 0.17), ('LLY', 0.11339), ('VZ', 0.0), ('JNJ', 0.0), ('EXC', 0.0), ('AMGN', 0.01765), ('MO', 0.06919), ('MDLZ', 0.0), ('PFE', 0.0), ('PG', 0.0), ('SO', 0.17), ('KO', 0.0)])
Expected annual return: 30.4%
Annual volatility: 7.8%
Sharpe Ratio: 3.75


/usr/local/lib/python3.10/dist-packages/pypfopt/efficient_frontier/efficient_frontier.py:441: UserWarning: The risk_free_rate provided to portfolio_performance is different to the one used by max_sharpe. Using the previous value.
  warnings.warn(


(0.3037867268643426, 0.07835661768273816, 3.749354369197885)

In [169]:
weight1=np.array(pd.DataFrame(cleaned_weights.values(), index = list(cleaned_weights),columns = ["weight"]))[:,0]

In [170]:
sd1 = np.sqrt(weight1@cov@weight1.T)
R1 = weight1.T@E_2/100
sharpe_ratio1 = (R1-0.01)/sd1
print("new_sharpe:",sharpe_ratio1)

new_sharpe: 3.7493208553679334


weight2 =   pd.read_excel("old_weight.xlsx",index_col=0)
weight2=np.array(weight2.loc[list(cleaned_weights),:])[:,0]

sd2 = np.sqrt(weight2@cov@weight2.T)
R2 = weight2.T@E_2/100
sharpe_ratio2 = (R2-0.01)/sd2
print("old_sharpe:",sharpe_ratio2)

In [171]:
weight=pd.DataFrame(cleaned_weights.values(), index = list(cleaned_weights),columns = ["weight"])
weight = weight[weight.weight >=0.030]
weight.weight = weight.weight/sum(weight.weight)
weight.to_csv("weights.csv")

See pyportfolioopt.readthedocs.io/en/latest/MeanVariance.html for more reference


In [172]:
if stock_type == "S&P100":
    OEX=pdr.get_data_yahoo("^OEX", start=start_swan_date, end=end_swan_date)["Adj Close"]

elif stock_type == "S&P500":
    OEX=pdr.get_data_yahoo("SPY", start=start_swan_date, end=end_swan_date)["Adj Close"]

stock_price=pdr.get_data_yahoo(list(weight.index), start=start_swan_date, end=end_swan_date)["Adj Close"]

[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  7 of 7 completed


In [173]:
stock_price.to_csv("tradeinfo.csv")
stock_price=stock_price[weight.index]
stock_price

Ticker,AMZN,WMT,COST,NEE,LLY,MO,SO
Date,,,,,,,
2020-02-18,107.783501,37.183620,297.414612,63.518692,133.254410,31.216713,58.356689
2020-02-19,108.511002,36.577515,298.310394,61.858879,133.084259,31.161488,57.692410
2020-02-20,107.654999,36.580620,299.289337,62.390388,134.189941,31.451433,58.701462
2020-02-21,104.798500,36.857258,297.322296,62.516495,134.227768,31.679237,58.011948
2020-02-24,100.464500,36.154789,289.629517,61.694477,130.551682,30.236448,57.011295
...,...,...,...,...,...,...,...
2021-03-16,154.593002,42.152634,312.725159,69.669754,183.833542,37.304352,53.824867
2021-03-17,156.786499,41.801868,314.578979,67.433090,179.817871,37.560890,53.341557
2021-03-18,151.399506,41.256065,308.644653,65.741722,177.121536,38.156979,53.077919


In [174]:
weight

,weight
AMZN,0.177059
WMT,0.177059
COST,0.101601
NEE,0.177059
LLY,0.118099
MO,0.072063
SO,0.177059


In [175]:
start=stock_price.loc["2020-02-19"]
end = stock_price.loc["2020-03-23"]
stock_drop=abs(end/start-1)
market_drop = abs(OEX.loc["2020-03-23"]/OEX.loc["2020-02-19"]-1)

In [176]:
beta = stock_drop/market_drop
1-beta.T@weight

weight    0.35608
dtype: float64

In [177]:
beta

Ticker
AMZN    0.390752
WMT     0.078002
COST    0.368082
NEE     1.062981
LLY     0.490480
MO      0.966755
SO      1.173178
dtype: float64

In [178]:
stock_drop

Ticker
AMZN    0.123209
WMT     0.024595
COST    0.116061
NEE     0.335171
LLY     0.154654
MO      0.304829
SO      0.369917
dtype: float64

In [179]:
rest=1
extra=0
for i, row in weight.iterrows():
  if row['weight']>0.17:
    rest=rest-row['weight']
    extra=extra+(row['weight']-0.17)
    # row['weight']=0.17

for i,row in weight.iterrows():
  if row['weight']<0.165:
    # row['weight']=row['weight']+row['weight']/rest*extra
    a=1

weight.to_csv("weights.csv")
weight

,weight
AMZN,0.177059
WMT,0.177059
COST,0.101601
NEE,0.177059
LLY,0.118099
MO,0.072063
SO,0.177059


In [180]:
portfolio_beta = portfolio_beta.loc[weight.index]
portfolio_beta

,Swan-3,Swan-2,Swan-1,Swan1,Swan2,Swan3,Swan4,Swan5,Swan6,Swan7,Swan8,Swan9,Swan10a,Swan10b
AMZN,NaN,NaN,NaN,NaN,NaN,1.584517,-0.445606,0.328167,0.123350,0.334033,1.561997,0.390752,1.440979,1.690943
WMT,0.897873,-1.424131,0.462878,1.059818,1.484265,0.762944,0.586029,0.461023,0.235666,0.982186,0.414455,0.078002,0.291725,0.710631
COST,NaN,NaN,NaN,1.052794,1.853865,1.404504,0.829818,0.976294,-0.015790,0.618966,0.918643,0.368083,0.536185,0.824934
NEE,0.253300,-0.134129,0.533954,0.579706,0.166846,-0.420653,0.414534,0.503894,0.415367,0.192033,-0.026635,1.062981,0.721034,0.894444
LLY,0.963250,0.201971,-0.343427,1.143233,0.898003,0.125934,0.746268,0.920577,0.280671,0.780342,0.061466,0.490481,0.281552,-0.331040
MO,0.308320,0.177528,0.542427,0.449382,0.538171,-0.274878,0.818853,0.448863,-0.137637,0.264024,1.030036,0.966755,-0.434630,-0.002853
SO,NaN,NaN,0.692450,0.735964,-0.130113,-0.265152,-0.138738,0.669547,-0.279810,0.055182,-0.091814,1.173178,-0.036447,0.004823


In [181]:
a=stock_price@weight
b=1-a.loc["2020-03-23"]/a.loc["2020-02-19"]
1-b/market_drop

weight    0.446273
dtype: float64

In [182]:
injected_result = pd.DataFrame(index = stock_price.index)
S1 = np.array(10000*weight.T)/np.array(stock_price.iloc[0,:]).T


In [183]:
portfolio_value = np.sum(np.array(weight.T) * stock_price,axis=1)
S1 = np.array(10000*weight.T)/np.array(stock_price.iloc[0,:]).T
injected_result["portfolio_value"]=stock_price@S1.T
S2 = 10000/float(OEX[0])
injected_result["oex"]=S2*OEX
injected_result.to_csv("Injected_result.csv")


In [184]:
temp=injected_result.loc[["2020-02-19","2020-03-23"]]
Portfolio_DD=abs(temp['portfolio_value'][1]/temp['portfolio_value'][0]-1)
DD=abs(temp['oex'][1]/temp['oex'][0]-1)
PR=1-(Portfolio_DD/DD)
print(PR)
temp

0.35787209669736464


,portfolio_value,oex
Date,,
2020-02-19,9916.944476,10051.749340
2020-03-23,7909.054653,6882.312479


In [185]:
allocation=pd.read_excel('allocation weight.xlsx',index_col='stock selected')
dfMVOstock=deepcopy(weight)
dfMVOstock['PR_SR']=''
dfMVOstock['SR']=''
for i in range(len(weight)):
    dfMVOstock.iloc[i,1]=preselected_stock.loc[weight.index[i],'PR_SR']
    dfMVOstock.iloc[i,2]=preselected_stock.loc[weight.index[i],'SR']
dfMVOstock_sort=dfMVOstock.sort_values(by='PR_SR',ascending=False)
postallocationweight=post_allocation(dfMVOstock_sort)
postallocationweight


,weight,PR_SR,SR,equal,proportion,adjusted proportion,direct
AMZN,0.177059,5.092006,87.203768,0.142857,0.425293,0.17,0.17
NEE,0.177059,2.300212,17.794796,0.142857,0.192118,0.17,0.17
WMT,0.177059,1.98281,13.22269,0.142857,0.165608,0.17,0.17
SO,0.177059,0.739287,1.838163,0.142857,0.061747,0.130001,0.17
LLY,0.118099,0.680906,1.559309,0.142857,0.05687,0.125125,0.17
MO,0.072063,0.661405,1.471271,0.142857,0.055242,0.123496,0.075
COST,0.101601,0.516312,0.896567,0.142857,0.043123,0.111378,0.075


In [186]:
#draw type E graph for postallocation
MVOweight=postallocationweight['weight']
equalweight=postallocationweight['equal']
proportionweight=postallocationweight['adjusted proportion']
directweight=postallocationweight['direct']
output(MVOweight).to_csv('no_post_allocation_result.csv')
output(equalweight).to_csv('equal_weight_result.csv')
output(proportionweight).to_csv('proportional_weight_result.csv')
output(directweight).to_csv('direct_weight_result.csv')

[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  7 of 7 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  7 of 7 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  7 of 7 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  7 of 7 completed


#